In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double to find contracts >= 1,000,000.
   3. TPRM STRING MAPPING: Maps TP engagements to AU IDs by checking if the 
      'TPRM_Units' string exists inside the 'owning_lob' or 'lob_using' columns.
      - Blocks blank mapping strings using NULLIF to prevent wildcard explosions.
      - Uses Databricks RLIKE with word boundaries (\b) for exact phrase matching.
   4. AGGREGATING (NUMERATOR & DENOMINATOR): Counts total DISTINCT engagements 
      (Denominator) and total DISTINCT engagements >= 1MM (Numerator).
   5. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Third_Party_Risk_Data AS (
    -- 2. Extract TP data, clean the contract amount string, and cast to double
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Engagements_By_AU AS (
    -- 3 & 4. Map engagements using the TPRM unit mapping table and calculate Num/Denom
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        -- Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT tp.EngagementNumber) AS Denominator,
        -- Distinct Engagements mapped to this AU that are >= 1 Million
        COUNT(DISTINCT CASE WHEN tp.Cleaned_Amount >= 1000000 THEN tp.EngagementNumber END) AS Numerator
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    
    -- FIXED JOIN: Uses Regex word boundaries and completely blocks blank mapping strings
    INNER JOIN Third_Party_Risk_Data tp
        ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
       AND (
           UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
            OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
       )
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 5. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.Mapped_AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 Drill-Down
   
   PURPOSE: Shows EVERY high-value third-party engagement (>= $1M), regardless of 
   whether the string mapped to an AU, or whether that AU exists in the Master List. 
   Useful for tracking unmapped mega-contracts.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Value_Engagements AS (
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        Contract_Amount AS Raw_Contract_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      -- Strictly filter for only the high-value contracts (Numerator targets)
      AND TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) >= 1000000
)

SELECT DISTINCT
    COALESCE(TRIM(CAST(map.Assessable_Unit_ID AS STRING)), 'UNMAPPED_ENGAGEMENT') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    hv.EngagementNumber,
    hv.Raw_Contract_Amount,
    hv.Cleaned_Amount,
    hv.owning_lob AS Raw_Col_K_owning_lob,
    hv.lob_using AS Raw_Col_L_lob_using,
    map.TPRM_Units AS Matched_Mapping_String
FROM High_Value_Engagements hv

-- Join to mapping table using the safe RLIKE logic
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
   AND (
       UPPER(hv.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
       OR UPPER(hv.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
   )
   
LEFT JOIN Master_AUs mast
    ON TRIM(CAST(map.Assessable_Unit_ID AS STRING)) = mast.BusinessID
ORDER BY BusinessID, hv.Cleaned_Amount DESC;